# barrier_drawdown_budget_overlay_v1

Notebook-owned execution-layer diagnostic for the active objective: keep the barrier-label return uplift while reducing key-window MaxDD below `15%`.

Governance:
- Canonical baseline: `runtime/production/card/paper_model_oos_risk_check_v1/paper_model_oos_risk_check_v1_soxl_daily.csv`.
- Direct comparison baseline: `runtime/model/card/barrier_action_label_purged_bc_v1/selected_paths.csv`, because barrier labels crossed the `+30pp` mean relative-return gate but failed drawdown (`36.58%` key-window MaxDD).
- Role: `diagnose`; fixed-path counterfactual only, no actor checkpoint and no runtime promotion.
- Changed layer: `execution` / risk-budget mapping. The actor/model/label path is fixed.
- Reusable code: `src/soxl_quant/app/drawdown_budget_overlay.py`; experiment-specific grid and conclusions stay here.
- Target windows: `wf_2022`, `wf_2023_2024`, `bad_C002_2025_rebound`, `bad_C003_2026_reentry`.
- Gate: mean relative-return delta vs production baseline `>= +30pp`, every key-window MaxDD `<= 15%`, and no-worse-window guard versus the production baseline.
- External references: Qlib (`https://github.com/microsoft/qlib`, `https://arxiv.org/abs/2009.11189`) for auditable workflow; CPPI/TIPP-style portfolio-insurance logic for drawdown-budget overlays; prior barrier/action-label notebook as the fixed signal source.

Closest historical success case:
- `risk_budget_residual_drawdown_guard_v1` reached strict OOS `relative_return=+30.46pp` and `MaxDD=14.64%`, but needed walk-forward validation.

Closest historical failure case:
- `barrier_action_label_purged_bc_v1` reached mean relative uplift `+32.02pp` but pushed `wf_2022` MaxDD to `36.58%`.

Why this is not a duplicate:
- It does not retrain, relabel, or resweep classifiers. It fixes the barrier-selected path and tests only whether an explicit realized-equity drawdown budget can turn return uplift into acceptable risk.


In [1]:
from __future__ import annotations

import json
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src.soxl_quant.app.budget_counterfactual import compute_metrics
from src.soxl_quant.app.drawdown_budget_overlay import DrawdownBudgetOverlayConfig, apply_drawdown_budget_overlay
from src.soxl_quant.app.purged_walkforward import single_position_frame

EXPERIMENT = "barrier_drawdown_budget_overlay_v1"
OUTPUT_DIR = Path("runtime/execution/card") / EXPERIMENT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_DAILY_CSV = Path("runtime/production/card/paper_model_oos_risk_check_v1/paper_model_oos_risk_check_v1_soxl_daily.csv")
BARRIER_SELECTION_JSON = Path("runtime/model/card/barrier_action_label_purged_bc_v1/selection.json")
BARRIER_PATHS_CSV = Path("runtime/model/card/barrier_action_label_purged_bc_v1/selected_paths.csv")
DAILY = pd.read_csv(BASELINE_DAILY_CSV, parse_dates=["datetime"]).sort_values("datetime").reset_index(drop=True)
PRICE = DAILY.set_index("datetime")["price"].astype(float)
BARRIER_PATHS = pd.read_csv(BARRIER_PATHS_CSV, parse_dates=["datetime"]).sort_values(["window", "datetime"]).reset_index(drop=True)
BARRIER_SELECTION = json.loads(BARRIER_SELECTION_JSON.read_text())
WINDOWS = {
    "wf_2022": ("2022-01-01", "2022-12-31"),
    "wf_2023_2024": ("2023-01-01", "2024-12-31"),
    "bad_C002_2025_rebound": ("2025-04-04", "2025-07-22"),
    "bad_C003_2026_reentry": ("2026-02-04", "2026-05-21"),
}
print({
    "baseline_rows": len(DAILY),
    "barrier_path_rows": len(BARRIER_PATHS),
    "barrier_decision": BARRIER_SELECTION.get("decision"),
    "barrier_mean_delta": BARRIER_SELECTION.get("selected_candidate", {}).get("mean_delta_relative_return"),
    "barrier_max_dd": BARRIER_SELECTION.get("selected_candidate", {}).get("max_key_window_dd"),
})

{'baseline_rows': 1605, 'barrier_path_rows': 902, 'barrier_decision': 'diagnostic_not_promoted', 'barrier_mean_delta': 32.01926899891025, 'barrier_max_dd': 36.576424413602325}


In [2]:
def score_positions(frame: pd.DataFrame, start: str, end: str, position_col: str = "position") -> dict:
    sub = frame[(frame["datetime"] >= start) & (frame["datetime"] <= end)].copy()
    return compute_metrics(single_position_frame(sub, position_col=position_col), PRICE.loc[start:end])


def baseline_table() -> pd.DataFrame:
    rows = []
    for window, (start, end) in WINDOWS.items():
        rows.append({"window": window, "variant": "baseline", **score_positions(DAILY, start, end)})
        barrier_sub = BARRIER_PATHS[BARRIER_PATHS["window"] == window].copy()
        rows.append({"window": window, "variant": "barrier_bc", **score_positions(barrier_sub, start, end)})
    return pd.DataFrame(rows)

BASE_WINDOWS = baseline_table()
BASE_WINDOWS[["window", "variant", "return", "relative_return", "max_dd", "avg_pos", "trades"]]

,window,variant,return,relative_return,max_dd,avg_pos,trades
0,wf_2022,baseline,-21.567182,65.020890,31.096849,0.248277,26
1,wf_2022,barrier_bc,-6.475865,80.112207,36.576424,0.350544,43
2,wf_2023_2024,baseline,209.141657,17.368148,30.668184,0.497231,70
3,wf_2023_2024,barrier_bc,307.364380,115.590871,24.260167,0.424908,96
4,bad_C002_2025_rebound,baseline,76.882481,-124.606649,16.324924,0.572851,9
5,bad_C002_2025_rebound,barrier_bc,91.645517,-109.843614,16.324924,0.582175,10
6,bad_C003_2026_reentry,baseline,17.799626,-217.582972,6.421473,0.161722,2
7,bad_C003_2026_reentry,barrier_bc,17.799626,-217.582972,6.421473,0.161722,2


In [3]:
def score_overlay_candidate(cfg: DrawdownBudgetOverlayConfig) -> tuple[dict, list[dict], dict[str, pd.DataFrame]]:
    comparison_rows = []
    path_by_window = {}
    deltas = []
    max_dds = []
    no_worse_windows = 0
    active_days = 0
    for window, (start, end) in WINDOWS.items():
        base = BASE_WINDOWS[(BASE_WINDOWS["window"] == window) & (BASE_WINDOWS["variant"] == "baseline")].iloc[0].to_dict()
        barrier = BASE_WINDOWS[(BASE_WINDOWS["window"] == window) & (BASE_WINDOWS["variant"] == "barrier_bc")].iloc[0].to_dict()
        barrier_sub = BARRIER_PATHS[BARRIER_PATHS["window"] == window].copy()
        overlay = apply_drawdown_budget_overlay(barrier_sub[["datetime", "position"]], PRICE.loc[start:end], cfg)
        overlay["window"] = window
        path_by_window[window] = overlay
        metrics = score_positions(overlay, start, end)
        delta_relative = metrics["relative_return"] - float(base["relative_return"])
        deltas.append(delta_relative)
        max_dds.append(metrics["max_dd"])
        if delta_relative >= -1e-9 and metrics["max_dd"] <= float(base["max_dd"]) + 1e-9:
            no_worse_windows += 1
        active_days += int(overlay["drawdown_budget_active"].sum())
        comparison_rows.append({"window": window, "variant": "baseline", **base})
        comparison_rows.append({"window": window, "variant": "barrier_bc", **barrier})
        comparison_rows.append({"window": window, "variant": "drawdown_budget", **metrics, "delta_relative_return": delta_relative})
    row = {
        "start_drawdown": float(cfg.start_drawdown),
        "full_drawdown": float(cfg.full_drawdown),
        "min_position": float(cfg.min_position),
        "trade_threshold": float(cfg.trade_threshold),
        "trade_lock_days": int(cfg.trade_lock_days),
        "sell_trade_threshold": cfg.sell_trade_threshold,
        "sell_trade_lock_days": cfg.sell_trade_lock_days,
        "mean_delta_relative_return": float(np.mean(deltas)),
        "min_delta_relative_return": float(np.min(deltas)),
        "max_key_window_dd": float(np.max(max_dds)),
        "no_worse_window_count": int(no_worse_windows),
        "active_days": int(active_days),
    }
    row["return_gate"] = bool(row["mean_delta_relative_return"] >= 30.0)
    row["maxdd_gate"] = bool(row["max_key_window_dd"] <= 15.0)
    row["no_worse_gate"] = bool(no_worse_windows == len(WINDOWS))
    row["full_gate"] = bool(row["return_gate"] and row["maxdd_gate"] and row["no_worse_gate"])
    row["diagnostic_score"] = float(row["mean_delta_relative_return"] - 2.0 * max(0.0, row["max_key_window_dd"] - 15.0) - 0.02 * active_days)
    return row, comparison_rows, path_by_window

In [4]:
rows = []
comparison_by_key = {}
path_by_key = {}
for start_dd, full_dd, min_position, trade_threshold in product(
    [0.04, 0.06, 0.08, 0.10, 0.12],
    [0.08, 0.10, 0.12, 0.15, 0.18],
    [0.0, 0.10, 0.20, 0.25, 0.35],
    [0.0, 0.03],
):
    if full_dd < start_dd:
        continue
    cfg = DrawdownBudgetOverlayConfig(
        start_drawdown=start_dd,
        full_drawdown=full_dd,
        min_position=min_position,
        trade_threshold=trade_threshold,
        trade_lock_days=3 if trade_threshold > 0 else 0,
        sell_trade_threshold=0.0,
        sell_trade_lock_days=0,
    )
    row, comparison_rows, paths = score_overlay_candidate(cfg)
    key = len(rows)
    row["candidate_key"] = key
    rows.append(row)
    comparison_by_key[key] = comparison_rows
    path_by_key[key] = paths

results = pd.DataFrame(rows).sort_values(["full_gate", "diagnostic_score", "mean_delta_relative_return"], ascending=[False, False, False]).reset_index(drop=True)
results.to_csv(OUTPUT_DIR / "drawdown_budget_grid_results.csv", index=False)
print("grid rows", len(results), "full gate passes", int(results["full_gate"].sum()))
results.head(20)[[
    "candidate_key", "start_drawdown", "full_drawdown", "min_position", "trade_threshold", "mean_delta_relative_return",
    "min_delta_relative_return", "max_key_window_dd", "return_gate", "maxdd_gate", "no_worse_gate", "full_gate", "active_days", "diagnostic_score",
]]

grid rows 220 full gate passes 0


,candidate_key,start_drawdown,full_drawdown,min_position,trade_threshold,mean_delta_relative_return,min_delta_relative_return,max_key_window_dd,return_gate,maxdd_gate,no_worse_gate,full_gate,active_days,diagnostic_score
0,219,0.12,0.18,0.35,0.03,20.770883,0.000000,34.194151,False,False,False,False,230,-22.217419
1,189,0.10,0.18,0.35,0.03,8.530383,0.000000,34.205377,False,False,False,False,284,-35.560370
2,159,0.10,0.10,0.35,0.03,1.084174,-18.856172,31.519853,False,False,False,False,304,-38.035532
3,217,0.12,0.18,0.25,0.03,3.119077,-13.405858,32.975181,False,False,False,False,263,-38.091285
4,158,0.10,0.10,0.35,0.00,-7.123966,-39.311416,28.511304,False,False,False,False,300,-40.146573
5,118,0.08,0.10,0.35,0.00,-6.248238,-38.497973,28.511304,False,False,False,False,374,-40.750845
6,169,0.10,0.12,0.35,0.03,0.325873,-28.871178,32.757550,False,False,False,False,338,-41.949228
7,199,0.12,0.12,0.35,0.03,0.282521,-27.455133,33.864613,False,False,False,False,271,-42.866705
8,99,0.06,0.18,0.35,0.03,5.146797,-5.470091,34.330594,False,False,False,False,481,-43.134391
9,108,0.08,0.08,0.35,0.00,-8.545112,-52.091905,28.511304,False,False,False,False,380,-43.167719


In [5]:
selected = results.iloc[0].to_dict()
selected_key = int(selected["candidate_key"])
selected_comparison = pd.DataFrame(comparison_by_key[selected_key])
selected_comparison.to_csv(OUTPUT_DIR / "selected_comparison.csv", index=False)
selected_paths = pd.concat(path_by_key[selected_key].values(), ignore_index=True)
selected_paths.to_csv(OUTPUT_DIR / "selected_paths.csv", index=False)

active_summary = selected_paths.groupby("window").agg(
    rows=("datetime", "count"),
    active_days=("drawdown_budget_active", "sum"),
    max_equity_drawdown=("equity_drawdown", "max"),
    min_cap=("drawdown_budget_cap", "min"),
    avg_base_position=("base_position", "mean"),
    avg_overlay_position=("position", "mean"),
).reset_index()
active_summary.to_csv(OUTPUT_DIR / "selected_active_summary.csv", index=False)

selection = {
    "experiment": EXPERIMENT,
    "decision": "candidate_requires_saved_walkforward" if bool(selected["full_gate"]) else "diagnostic_not_promoted",
    "reason": (
        "drawdown budget overlay clears notebook return/drawdown/no-worse gates but still needs saved walk-forward validation"
        if bool(selected["full_gate"])
        else "drawdown budget overlay could not keep +30pp mean uplift while forcing every key-window MaxDD <= 15%"
    ),
    "baseline_daily_csv": str(BASELINE_DAILY_CSV),
    "fixed_signal_selection_json": str(BARRIER_SELECTION_JSON),
    "fixed_signal_paths_csv": str(BARRIER_PATHS_CSV),
    "grid_results_csv": str(OUTPUT_DIR / "drawdown_budget_grid_results.csv"),
    "selected_comparison_csv": str(OUTPUT_DIR / "selected_comparison.csv"),
    "selected_paths_csv": str(OUTPUT_DIR / "selected_paths.csv"),
    "selected_active_summary_csv": str(OUTPUT_DIR / "selected_active_summary.csv"),
    "selected_candidate": {k: (None if pd.isna(v) else v) for k, v in selected.items() if isinstance(v, (int, float, str, bool)) or pd.isna(v)},
    "barrier_reference": BARRIER_SELECTION.get("selected_candidate", {}),
    "next_path": (
        "package for saved-actor walk-forward validation before any promotion claim"
        if bool(selected["full_gate"])
        else "execution drawdown budget alone still trades away too much return or misses the <=15% MaxDD gate; next step should use regime/state-conditioned budget activation rather than a global realized-equity cap"
    ),
}
(OUTPUT_DIR / "selection.json").write_text(json.dumps(selection, indent=2, ensure_ascii=True), encoding="utf-8")
selected_comparison[["window", "variant", "return", "relative_return", "max_dd", "avg_pos", "trades"]].sort_values(["window", "variant"])

,window,variant,return,relative_return,max_dd,avg_pos,trades
7,bad_C002_2025_rebound,barrier_bc,91.645517,-109.843614,16.324924,0.582175,10
6,bad_C002_2025_rebound,baseline,76.882481,-124.606649,16.324924,0.572851,9
8,bad_C002_2025_rebound,drawdown_budget,91.645517,-109.843614,16.324924,0.582175,10
10,bad_C003_2026_reentry,barrier_bc,17.799626,-217.582972,6.421473,0.161722,2
9,bad_C003_2026_reentry,baseline,17.799626,-217.582972,6.421473,0.161722,2
11,bad_C003_2026_reentry,drawdown_budget,17.799626,-217.582972,6.421473,0.161722,2
1,wf_2022,barrier_bc,-6.475865,80.112207,36.576424,0.350544,43
0,wf_2022,baseline,-21.567182,65.020890,31.096849,0.248277,26
2,wf_2022,drawdown_budget,-8.311056,78.277016,34.194151,0.247465,39
4,wf_2023_2024,barrier_bc,307.364380,115.590871,24.260167,0.424908,96


In [6]:
active_summary

,window,rows,active_days,max_equity_drawdown,min_cap,avg_base_position,avg_overlay_position
0,bad_C002_2025_rebound,74,6,0.163249,0.531467,0.582175,0.582175
1,bad_C003_2026_reentry,75,0,0.064215,1.000000,0.161722,0.161722
2,wf_2022,251,168,0.341942,0.350000,0.350544,0.247465
3,wf_2023_2024,502,56,0.215202,0.350000,0.424908,0.415546


In [7]:
best = results.iloc[0]
barrier_best = BARRIER_SELECTION.get("selected_candidate", {})
summary = {
    "candidate_count": int(len(results)),
    "full_gate_pass_count": int(results["full_gate"].sum()),
    "best_mean_delta_relative_return": float(best["mean_delta_relative_return"]),
    "best_min_delta_relative_return": float(best["min_delta_relative_return"]),
    "best_max_key_window_dd": float(best["max_key_window_dd"]),
    "best_active_days": int(best["active_days"]),
    "barrier_reference_mean_delta": barrier_best.get("mean_delta_relative_return"),
    "barrier_reference_max_key_window_dd": barrier_best.get("max_key_window_dd"),
}
conclusion = f"""## Runtime Conclusion

Decision: `{selection['decision']}`.

Screened `{summary['candidate_count']}` fixed-path drawdown-budget overlays on the barrier-selected signal; full gate pass count `{summary['full_gate_pass_count']}`. Best overlay mean relative-return delta `{summary['best_mean_delta_relative_return']:.2f}pp`, min window delta `{summary['best_min_delta_relative_return']:.2f}pp`, max key-window DD `{summary['best_max_key_window_dd']:.2f}%`, active days `{summary['best_active_days']}`.

Direct comparison: the fixed barrier signal had mean relative-return delta `{summary['barrier_reference_mean_delta']:.2f}pp` and max key-window DD `{summary['barrier_reference_max_key_window_dd']:.2f}%`. The overlay is useful only if it keeps the return gate while materially lowering key-window DD.

Mechanism readout: this is a fixed actor/signal counterfactual. If a global realized-equity cap fails, the remaining issue is not classifier family or action label alone; the risk budget needs regime/state-conditioned activation so it cuts the 2022 loss path without suppressing rebound participation.

Keep/drop: keep `drawdown_budget_overlay.py` as reusable fixed-path infrastructure. Do not promote this overlay from notebook replay. Promote only after saved walk-forward validation if the notebook gate is clean.

Artifacts: `drawdown_budget_grid_results.csv`, `selected_comparison.csv`, `selected_paths.csv`, `selected_active_summary.csv`, and `selection.json`.
"""
display(Markdown(conclusion))
print(json.dumps(summary, indent=2))

## Runtime Conclusion

Decision: `diagnostic_not_promoted`.

Screened `220` fixed-path drawdown-budget overlays on the barrier-selected signal; full gate pass count `0`. Best overlay mean relative-return delta `20.77pp`, min window delta `0.00pp`, max key-window DD `34.19%`, active days `230`.

Direct comparison: the fixed barrier signal had mean relative-return delta `32.02pp` and max key-window DD `36.58%`. The overlay is useful only if it keeps the return gate while materially lowering key-window DD.

Mechanism readout: this is a fixed actor/signal counterfactual. If a global realized-equity cap fails, the remaining issue is not classifier family or action label alone; the risk budget needs regime/state-conditioned activation so it cuts the 2022 loss path without suppressing rebound participation.

Keep/drop: keep `drawdown_budget_overlay.py` as reusable fixed-path infrastructure. Do not promote this overlay from notebook replay. Promote only after saved walk-forward validation if the notebook gate is clean.

Artifacts: `drawdown_budget_grid_results.csv`, `selected_comparison.csv`, `selected_paths.csv`, `selected_active_summary.csv`, and `selection.json`.


{
  "candidate_count": 220,
  "full_gate_pass_count": 0,
  "best_mean_delta_relative_return": 20.770882542072474,
  "best_min_delta_relative_return": 0.0,
  "best_max_key_window_dd": 34.19415076349897,
  "best_active_days": 230,
  "barrier_reference_mean_delta": 32.01926899891025,
  "barrier_reference_max_key_window_dd": 36.576424413602325
}
